In [ ]:
# Interfaz
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import numpy as np
import scipy.signal as sig
import NNMF_GIF2 as CONMF
import NNMF_GIF as NMF
import Funciones as Funciones
import matplotlib.pyplot as plt
from numpy.polynomial import Polynomial
import pandas as pd
%config InlineBackend.figure_format = 'svg'
%matplotlib widget
#========================================================================================================================================
# Variables generales
#========================================================================================================================================
cmap = Funciones.parula_map
color_ACC = 'blue'
color_EGG = 'orange'
color_dEGG = 'g'
color_OVV = 'magenta'
color_speech = 'black'
legend_loc = 'upper right'
color1 = 'red'
color2 = 'blue'
color3 = 'k'
color_notch = 'green'
color_lpc  = 'orange'
color_flujo = 'k'
line_width = 1.5
#========================================================================================================================================
# Función principal
#========================================================================================================================================
def ejecutar(b):
    #==============================================================================================================================================================
    plt.close('all')
    archivo = Path(fc.selected)
    nombre_archivo = archivo.name

    df = pd.read_csv(fc.selected, sep=',')
    global tiempos
    tiempos      = df['t_ms'].values
    tipo_señal  = nmf_señal.value
    señal = df[tipo_señal].values
    señal = Funciones.Detrend(tiempos,señal)
    if tipo_señal == "OVV":
        OVV_filtrada_pb_, b_pb_, a_pb_ = Funciones.LP_Filter(señal, fs=8000, fc=fc_filtro.value, order=orden_filtro.value)
        OVV_filtrada_pb_pa_, _, _      = Funciones.HP_Filter(OVV_filtrada_pb_, fs=8000)
        señal = OVV_filtrada_pb_pa_
    señal = señal-np.mean(señal)
    señal = señal/max(abs(señal))
    EGG  =  df['EGG'].values
    dEGG = np.gradient(EGG)

    linewidth   = 2
    fs_r = 8000
    to = 0
    tf = 50

    min_dEGG = min(dEGG)
    max_dEGG = max(dEGG)
    lim_inf_dEGG = min_dEGG*1.5
    lim_sup_dEGG = max_dEGG*1.5
    global resultado_NMF_speech
    resultado_NMF = NMF.NNMF_GIF(señal=señal,
                                        dEGG = dEGG,
                                        tiempos=tiempos,
                                        fs=fs_r,
                                        norma=nmf_norma.value,
                                        pre_iteraciones=nmf_pre_iteraciones.value,
                                        iteraciones=nmf_iteraciones.value,
                                        p_filtro=nmf_p_filtro.value,
                                        alfa=nmf_alfa.value,
                                        beta=nmf_beta.value,
                                        pre_enfasis=nmf_pre_enfasis.value,
                                        duracion=nmf_duracion_ventana.value,
                                        guardar_proceso=nmf_guardar_proceso.value,
                                        ventana = nmf_ventana.value
                                        )
    resultado_NMF.flujo1 = Funciones.Detrend(resultado_NMF.tiempos_gve,resultado_NMF.flujo1)
    resultado_NMF.dflujo1 = np.gradient(resultado_NMF.flujo1)
    resultado_NMF.flujo2 = Funciones.Detrend(resultado_NMF.tiempos_gve,resultado_NMF.flujo2)
    resultado_NMF.dflujo2 = np.gradient(resultado_NMF.flujo2)

    with tab_señales_originales:
        clear_output(wait=True)
        fig = plt.figure(figsize=(10,5))
        fig.canvas.header_visible = False
        widths  = [1]
        heights = [0.5,1,]
        gs  = fig.add_gridspec(2, 1,width_ratios=widths,height_ratios=heights,hspace=0.35)

        # Señal
        ax1 = fig.add_subplot(gs[0])
        ax1.plot(tiempos,señal,color='k',linewidth=line_width)
        ax1.set_xlabel("Tiempo[ms]")
        ax1.set_title(tipo_señal)
        # Espectrograma
        ax2 = fig.add_subplot(gs[1])
        #========================= Espectrograma speech
        im2 = ax2.imshow(resultado_NMF.espectrograma,
                        cmap=cmap,
                        origin='lower',
                        aspect='auto',
                        extent=[resultado_NMF.to_espectrograma, resultado_NMF.tf_espectrograma,
                                resultado_NMF.fo_espectrograma, resultado_NMF.ff_espectrograma])
        ax2.set_ylabel('Frecuencia [Hz]')
        ax2.set_xlabel("Tiempo [ms]")

        # ------------- dEGG
        ax2_ = ax2.twinx()
        ax2_.plot(tiempos,dEGG,color='white', label=r"$dEGG$", linewidth=line_width)
        ax2_.legend(loc=legend_loc)
        ax2_.set_xlim(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma)
        ax2_.set_ylim([lim_inf_dEGG,lim_sup_dEGG])
        ax2_.set_xticks(np.arange(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma, resultado_NMF.duracion_ventana))
        ax2_.set_yticks([])
        plt.show()

    with tab_resultados_frecuencia:
        clear_output(wait=True)
        fig = plt.figure(figsize=(10, 13)) 
        fig.canvas.header_visible = False
        fig.subplots_adjust(top=0.97)  

        widths  = [1,0.5]
        heights = [1, 1, 0.5, 0.5]  # ajustar proporciones
        gs = fig.add_gridspec(4, 2,
                            width_ratios=widths,
                            height_ratios=heights,
                            wspace=0.15,
                            hspace=0.35)
        # Espectrograma
        ax1 = fig.add_subplot(gs[0,:])
        # Espectrograma aproximado
        ax2 = fig.add_subplot(gs[1,:])
        # Activacion
        ax3 = fig.add_subplot(gs[2,:])
        # Resp frecuencia y polos y ceros
        ax4 = fig.add_subplot(gs[3,0])
        ax5 = fig.add_subplot(gs[3,1])

        #========================= Espectrograma 
        im1 = ax1.imshow(resultado_NMF.espectrograma,
                        cmap=cmap,
                        origin='lower',
                        aspect='auto',
                        extent=[resultado_NMF.to_espectrograma, resultado_NMF.tf_espectrograma,
                                resultado_NMF.fo_espectrograma, resultado_NMF.ff_espectrograma])
        ax1.set_ylabel('Frecuencia [Hz]')
        ax1.set_xlabel("Tiempo [ms]")
        bbox = ax1.get_position()
        width = 0.015
        eps = 0.01
        cax = fig.add_axes([bbox.x1 + eps, bbox.y0, width, bbox.height])
        fig.colorbar(im1, cax=cax)
        # ------------- dEGG
        ax1_ = ax1.twinx()
        ax1_.plot(tiempos,dEGG,color='white', label=r"$dEGG$", linewidth=line_width)
        ax1_.legend(loc=legend_loc)
        ax1_.set_xlim(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma)
        ax1_.set_ylim([lim_inf_dEGG,lim_sup_dEGG])
        ax1_.set_xticks(np.arange(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma, resultado_NMF.duracion_ventana))
        ax1_.set_yticks([])
        #========================= Espectrograma aproximado sincronizado
        im2 = ax2.imshow(resultado_NMF.espectrograma_aprox,
                        cmap=cmap,
                        origin='lower',
                        aspect='auto',
                        extent=[resultado_NMF.to_espectrograma, resultado_NMF.tf_espectrograma,
                                resultado_NMF.fo_espectrograma, resultado_NMF.ff_espectrograma])
        ax2.set_ylabel('Frecuencia [Hz]')
        ax2.set_xlabel("Tiempo [ms]")
        bbox = ax2.get_position()
        width = 0.015
        eps = 0.01
        cax = fig.add_axes([bbox.x1 + eps, bbox.y0, width, bbox.height])
        fig.colorbar(im2, cax=cax)
        # ------------- dEGG
        ax2_ = ax2.twinx()
        ax2_.plot(tiempos,dEGG,color='white', label=r"$dEGG$", linewidth=line_width)
        ax2_.legend(loc=legend_loc)
        ax2_.set_xlim(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma)
        ax2_.set_ylim([lim_inf_dEGG,lim_sup_dEGG])
        ax2_.set_xticks(np.arange(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma, resultado_NMF.duracion_ventana))
        ax2_.set_yticks([])

        #========================= Funciones de activación
        ax3.plot(resultado_NMF.tiempos_espectrograma,resultado_NMF.H1,color=color1,label=r'$H_1$')
        ax3.plot(resultado_NMF.tiempos_espectrograma,resultado_NMF.H2,color=color2,label=r'$H_2$')
        ax3.set_xlim(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma)
        ax3.set_xticks(np.arange(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma,resultado_NMF.duracion_ventana))
        ax3.set_xticks(np.arange(resultado_NMF.to_espectrograma,resultado_NMF.tf_espectrograma,resultado_NMF.duracion_ventana/2),minor=True)
        ax3.grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
        ax3.legend(loc=legend_loc)
        ax3.set_xlabel('Tiempo [ms]')
        #========================= Respuesta en frecuencia/polos-ceros speech
        # ------------- Respuesta en frecuencia
        theta = np.linspace(0, 2*np.pi, 200)
        fftseñal, fseñal = Funciones.FFT(señal, fs_r)
        fw1,hw1 = sig.freqz([1],resultado_NMF.a_W1,fs=fs_r)
        fw2,hw2 = sig.freqz([1],resultado_NMF.a_W2,fs=fs_r)
        ax4.plot(fw1, 10*np.log10(np.abs(hw1)), label=r'$W_1$', color=color1)
        ax4.plot(fw2, 10*np.log10(np.abs(hw2)), label=r'$W_2$', color=color2)
        ax4.plot(fseñal, 10*np.log10(np.abs(fftseñal)), label='speech', color=color3, zorder=-1)
        ax4.set_xlim(0,fs_r/2)
        ax4.set_xticks(np.arange(0, fs_r/2, 100), minor=True)
        ax4.grid(True, linestyle='dashed', color='gray', alpha=0.3, which='both')
        ax4.set_xlabel('Frecuencia [Hz]')
        ax4.set_ylabel('[dB]')
        ax4.set_ylim([-30,20])
        ax4.legend(loc='upper right')
        # ------------- Polos y ceros   
        ax5.plot(np.cos(theta), np.sin(theta), 'k--', alpha=1)
        ax5.plot(0.8*np.cos(theta), 0.8*np.sin(theta), 'k--', alpha=0.5)
        ax5.yaxis.tick_right()
        ax5.yaxis.set_label_position("right")
        ax5.scatter(np.real(resultado_NMF.ceros_W1), np.imag(resultado_NMF.ceros_W1),s=80, facecolors='none', edgecolors=color1, label='Ceros')
        ax5.scatter(np.real(resultado_NMF.polos_W1), np.imag(resultado_NMF.polos_W1),s=80, marker='x', color=color1, label='Polos')
        ax5.scatter(np.real(resultado_NMF.ceros_W2), np.imag(resultado_NMF.ceros_W2),s=80, facecolors='none', edgecolors=color2, label='Ceros')
        ax5.scatter(np.real(resultado_NMF.polos_W2), np.imag(resultado_NMF.polos_W2),s=80, marker='x', color=color2, label='Polos')
        ax5.set_xticks(np.arange(-1.25,1.25,0.25), minor=True)
        ax5.set_yticks(np.arange(-1.25,1.25,0.25), minor=True)
        ax5.set_ylabel(r"$Im\{z\}$")
        ax5.set_xlabel(r"$Re\{z\}$")
        ax5.grid(True, linestyle='dashed', color='gray', alpha=0.3)


        plt.show()

    with tab_errores:
        clear_output(wait=True)
        clear_output(wait=True)  
        fig, ax = plt.subplots(4, 1, figsize=(10, 12))
        fig.subplots_adjust(hspace=0.35)
        fig.canvas.header_visible = False
        fig.subplots_adjust(top=0.97) 
        ax[0].plot(resultado_NMF.error_NNMF, color='orange', label='Error')
        ax0_ = ax[0].twinx()
        ax0_.plot(resultado_NMF.dif_error_X*100, color='green', label='Δ Error')
        ax0_.set_ylim([-0.5,2])
        ax[0].set_ylabel('Error')
        ax0_.set_ylabel('Δ Error [%]')
        ax[0].set_xlabel('Iteración n°')
        h1, l1 = ax[0].get_legend_handles_labels()
        h2, l2 = ax0_.get_legend_handles_labels()
        ax[0].legend(h1 + h2, l1 + l2, loc=legend_loc,)

        ax[1].plot(resultado_NMF.dif_error_H1*100, color=color1, label='Δ H1')
        ax[1].plot(resultado_NMF.dif_error_H2*100, color=color2, label='Δ H2')
        ax[1].set_ylabel('Δ [%]')
        ax[1].set_ylim([0,2])
        ax[1].legend(loc=legend_loc)
        ax[1].set_xlabel('Iteración n°')

        ax[2].plot(resultado_NMF.dif_error_W1*100, color=color1, label='Δ W1')
        ax[2].plot(resultado_NMF.dif_error_W2*100, color=color2, label='Δ W2')
        ax[2].set_ylabel('Δ [%]')
        ax[2].set_ylim([0,1])
        ax[2].legend(loc=legend_loc)
        ax[2].set_xlabel('Iteración n°')

        ax[3].plot(resultado_NMF.dif_error_X*100, color='green', label='Δ Error')
        ax[3].plot(resultado_NMF.dif_error_H1*100, color=color1, label='Δ H1')
        ax[3].plot(resultado_NMF.dif_error_H2*100, color=color2, label='Δ H2')
        ax[3].set_ylim([0,2])
        ax[3].legend(loc=legend_loc)

        ax[3].plot(resultado_NMF.dif_error_W1*100, label='Δ W1')
        ax[3].plot(resultado_NMF.dif_error_W2*100, label='Δ W2')

        ax[3].set_ylabel('Δ [%]')
        ax[3].set_ylim([0,1])
        ax[3].legend(loc=legend_loc)
        ax[3].set_xlabel('Iteración n°')
        plt.show()

    with tab_criterio_seleccion:
        clear_output(wait=True)
        fig,ax = plt.subplots(2,figsize=(10,5))
        fig.subplots_adjust(hspace=0.35)
        fig.canvas.header_visible = False
        fig.subplots_adjust(top=0.97) 
        ax[0].plot(resultado_NMF.tiempos_gve,resultado_NMF.error_predic1,color=color1,label=r'$e_1(t)$');
        ax[0].plot(resultado_NMF.tiempos_gve,resultado_NMF.error_predic2,color=color2,label=r'$e_2(t)$');
        ax[0].set_xlim([to,tf]);
        ax[0].set_xticks(np.arange(to,tf,2.5),minor=True)
        ax[0].tick_params(axis='x', which='both')
        ax[0].grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
        ax[0].set_xlabel('Tiempo [ms]');
        ax[0].legend(loc=legend_loc)

        ax[1].plot(resultado_NMF.tiempos_eH,resultado_NMF.e1H1,color=color1,label=r"$H_1(t)|e_1(t)|$"+"  ~ " + r"$\frac{\sum H_1(t)|e_1(t)|}{\sum H_1(t)}=$"+f"{round(resultado_NMF.suma_e1H1,6)}")
        ax[1].plot(resultado_NMF.tiempos_eH,resultado_NMF.e2H2,color=color2,label=r"$H_2(t)|e_2(t)|$"+"  ~ " + r"$\frac{\sum H_2(t)|e_2(t)|}{\sum H_2(t)}=$"+f"{round(resultado_NMF.suma_e2H2,6)}")
        ax[1].set_xlim([to,tf]);
        ax[1].set_xticks(np.arange(to,tf,2.5),minor=True)
        ax[1].tick_params(axis='x', which='both')
        ax[1].grid(True,linestyle='dashed',color='gray',alpha=0.3,which='both')
        ax[1].set_xlabel('Tiempo [ms]');
        ax[1].legend(loc=legend_loc)

        plt.show()

    with tab_flujos_funciones:
        clear_output(wait=True)
        fig,ax = plt.subplots(2,1,figsize=(10,5))
        #fig.suptitle(nombre_archivo.split(".txt")[0] + " | Norma: "+norma, fontsize=12)
        plt.subplots_adjust(hspace=0.25)
        fig.canvas.header_visible = False
        fig.subplots_adjust(top=0.97) 
        ax[0].plot(resultado_NMF.tiempos_gve,resultado_NMF.flujo1,color=color1,linewidth=line_width,label=r"$u_{g1}(t)$")
        ax[0].plot(resultado_NMF.tiempos_gve,resultado_NMF.flujo2,color=color2,linewidth=line_width,label=r"$u_{g2}(t)$")
        ax[0].set_xlabel('Tiempo [ms]');
        ax[0].legend(loc=legend_loc)

        ax[1].plot(resultado_NMF.tiempos_gve,resultado_NMF.dflujo1,color=color1,linewidth=line_width,label=r"$v_{g1}(t)$")
        ax[1].plot(resultado_NMF.tiempos_gve,resultado_NMF.dflujo2,color=color2,linewidth=line_width,label=r"$v_{g2}(t)$")
        ax[1].set_xlabel('Tiempo [ms]')
        ax[1].legend(loc=legend_loc)
        plt.show()

#========================================================================================================================================
# Títulos de las secciones
#========================================================================================================================================
titulo_archivo = widgets.HTML(
    value="<b>Archivo</b>"
)
titulo_CONMF = widgets.HTML(
    value="""
    <div style='width:100%; font-size:36px; font-weight:bold; background-color:#8080ff; color:white; padding:10px;'>
        CO-NMF
    </div>
    """
)
titulo_NMF = widgets.HTML(
    value="""
    <div style='width:100%; font-size:36px; font-weight:bold; background-color:#ff7878; color:white; padding:10px;'>
        NMF
    </div>
    """
)
subtitulo_espectrograma = widgets.HTML(
    value="<b>Espectrograma</b>"
)
subtitulo_señal= widgets.HTML(
    value="<b>Señal</b>"
)

subtitulo_nmf= widgets.HTML(
    value="<b>NNMF</b>"
)
subtitulo_filtro_ovv= widgets.HTML(
    value="<b>Filtro PB OVV</b>"
)
subtitulo_gif= widgets.HTML(
    value="<b>GIF</b>"
)
linea_divisoria = widgets.HTML(
    value="<hr style='border:1px solid #ccc; margin:10px 0;'>"
)
#========================================================================================================================================
# Archivo y botón
#========================================================================================================================================

fc = FileChooser('Señales VPLAB')
fc.layout = widgets.Layout(width='auto')

boton_ejecutar = widgets.Button(
    description="Ejecutar",
    button_style='success'
)
grid_archivo_boton = widgets.GridBox(
    children=[fc, boton_ejecutar],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_gap='10px 50px', 
        width='60%'
    )
)
#========================================================================================================================================
# Elementos de NMF
#========================================================================================================================================
nmf_señal = widgets.Dropdown(
    options=['MBK','ACC','OVV'],
    value='MBK',
    description='Señal:',
    layout=widgets.Layout(width='200px')
)
nmf_norma = widgets.Dropdown(
    options=['frobenius','itakura-saito'],
    value='frobenius',
    description='Norma:',
    layout=widgets.Layout(width='300px')
)
nmf_ventana = widgets.Dropdown(
    options=['Rectangular','Hanning','Blackman','Hamming'],
    value='Rectangular',
    description='Ventana:',
    layout=widgets.Layout(width='200px')
)
nmf_duracion_ventana = widgets.BoundedFloatText(
    value=5.5,
    min=1.0,
    max=20.0,
    step=0.5,
    description='Duración ventana (ms):',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
nmf_desplazamiento = widgets.BoundedIntText(
    value=1,
    min=1,
    max=10,
    step=1,
    description='hop:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px'),
)
nmf_pre_iteraciones = widgets.BoundedIntText(
    value=0,
    min=0,
    max=100,
    step=1,
    description='Pre-iteraciones:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='150px')
)
nmf_iteraciones = widgets.BoundedIntText(
    value=250,
    min=1,
    max=50000,
    step=1,
    description='Iteraciones:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='150px')
)
nmf_p_filtro = widgets.BoundedIntText(
    value=12,
    min=2,
    max=18,
    step=1,
    description='Orden del filtro del tracto:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='200px')
)
nmf_alfa = widgets.BoundedFloatText(
    value=0.96,
    min=0.0,
    max=1.0,
    step=0.01,
    description='α:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='75px')
)
nmf_beta = widgets.BoundedFloatText(
    value=0.55,
    min=0.1,
    max=2.0,
    step=0.05,
    description='β:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)

nmf_pre_enfasis= widgets.Checkbox(
    value=True,
    description='Pre-énfasis'
)

nmf_guardar_proceso = widgets.Checkbox(
    value=True,
    description='Guardar proceso'
)
nmf_grid_señal = widgets.GridBox(
    children=[nmf_señal],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',  
        grid_gap='0px 5px',
        width='40%'
    )
)

nmf_grid_espectrograma = widgets.GridBox(
    children=[nmf_ventana, nmf_duracion_ventana, nmf_desplazamiento,nmf_pre_enfasis,nmf_beta],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',  
        width='70%'
    )
)
nmf_grid_nmf = widgets.GridBox(
    children=[nmf_norma, nmf_pre_iteraciones, nmf_iteraciones],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',  
        width='70%'
    )
)
nmf_grid_gif = widgets.GridBox(
    children=[nmf_p_filtro, nmf_alfa],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',  
        width='40%'
    )
)
orden_filtro   = widgets.BoundedIntText(
    value=10, min=2, max=20, step=1,
    description='Orden del filtro pasa bajos:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='210px')
)
fc_filtro = widgets.BoundedIntText(
    value=1100, min=200, max=2000, step=10,
    description='Frecuencia de corte [Hz]:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
nmf_grid_filtro_ovv= widgets.GridBox(
    children=[orden_filtro, fc_filtro],
    layout=widgets.Layout(
        display='grid',
        grid_auto_flow='column',
        grid_auto_columns='auto',
        grid_gap='0px 10px',  
        width='40%'
    )
)
#========================================================================================================================================
# Ensamble de la estructura de la intefaz
#========================================================================================================================================
estructura_interfaz = widgets.VBox([
    titulo_archivo,
    grid_archivo_boton,
    linea_divisoria,
    titulo_NMF,
    linea_divisoria,
    subtitulo_señal,
    nmf_grid_señal,
    linea_divisoria,
    subtitulo_filtro_ovv,
    nmf_grid_filtro_ovv,
    linea_divisoria,
    subtitulo_espectrograma,
    nmf_grid_espectrograma,
    linea_divisoria,
    subtitulo_nmf,
    nmf_grid_nmf,
    linea_divisoria,
    subtitulo_gif,
    nmf_grid_gif,
    linea_divisoria,
])
#========================================================================================================================================
# Configuración de las pestañas
#========================================================================================================================================
tab_señales_originales = widgets.Output()
tab_resultados_frecuencia = widgets.Output()
tab_errores = widgets.Output()
tab_criterio_seleccion = widgets.Output()
tab_flujos_funciones = widgets.Output()
tab_funciones_gloticas = widgets.Output()

tab = widgets.Tab(children=[
    estructura_interfaz,
    tab_señales_originales,
    tab_resultados_frecuencia,
    tab_errores,
    tab_criterio_seleccion,
    tab_flujos_funciones
])
tab.set_title(0,'Configuración')
tab.set_title(1,'Señales originales')
tab.set_title(2,'Frecuencia')
tab.set_title(3,'Errores')
tab.set_title(4,'Selección')
tab.set_title(5,'Flujos y funciones')
boton_ejecutar.on_click(ejecutar)
display(tab)